## 🎯 Learning Objectives
* Understand the core concepts and benefits of multi-agent orchestration in AI systems.
* Learn how to design and implement multi-agent workflows using LangChain's advanced features, specifically LangGraph.
* Develop practical skills in defining specialized agents, their tools, and orchestrating their interactions to solve complex problems.
* Analyze the performance trade-offs and identify suitable use cases for multi-agent architectures.


## Multi-agent Orchestration with LangChain

Imagine trying to produce a blockbuster movie with just one person. They'd have to write the script, direct the actors, manage the budget, shoot the scenes, edit the footage, and market the film—all by themselves. It's an impossible task, leading to burnout and a low-quality output. Instead, a film studio employs a specialized team: a screenwriter, a director, a cinematographer, an editor, a marketing team, and so on. Each expert focuses on their domain, collaborating under the guidance of a producer or director to achieve a common goal.

This analogy perfectly describes the power of **multi-agent orchestration** in AI. Instead of relying on a single, monolithic AI agent to tackle every aspect of a complex problem, we design a system where multiple specialized AI agents work together. Each agent is equipped with specific skills, tools, and a defined role, contributing its expertise to a larger workflow.

### Why Multi-Agent Orchestration?

1.  **Specialization**: Agents can be highly optimized for particular tasks (e.g., one agent for research, another for writing, a third for code generation). This leads to better performance and accuracy in each sub-task.
2.  **Modularity**: The system becomes easier to design, debug, and maintain. If one agent needs an update or a new tool, it doesn't necessarily impact the others.
3.  **Robustness**: If one agent fails or struggles with a specific input, the orchestration layer can potentially re-route, retry, or engage another agent, making the system more resilient.
4.  **Scalability**: Complex problems can be broken down into smaller, manageable sub-problems, allowing for more efficient processing and parallel execution where applicable.
5.  **Emergent Behavior**: The interaction between specialized agents can lead to more sophisticated and intelligent outcomes than any single agent could achieve alone.

### LangChain's Role in Orchestration

LangChain, especially with its `LangGraph` library, provides a robust framework for building and managing these multi-agent systems. `LangGraph` allows us to define agents as nodes in a graph and specify the flow of information and control between them. This enables:

*   **Sequential Chains**: Agents pass information from one to the next in a predefined order.
*   **Conditional Routing**: An agent's output can determine which subsequent agent or path to take.
*   **Cycles and Loops**: Agents can revisit previous steps or iterate on tasks until a condition is met (e.g., an editor refining content until it meets quality standards).
*   **Human-in-the-Loop**: Integration points where human feedback or intervention can guide the agent workflow.

In this lesson, we'll build a simple multi-agent system using LangGraph to demonstrate how specialized agents can collaborate to achieve a more complex task, such as generating a well-researched and structured article.


In [ ]:
import os
from typing import List, Dict, Any, TypedDict

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_community.tools import TavilySearchResults
from langchain.agents import AgentExecutor, create_openai_functions_agent

from langgraph.graph import StateGraph, END

# --- 1. Configuration and API Keys ---
# Ensure your OpenAI API key is set as an environment variable
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"
# os.environ["TAVILY_API_KEY"] = "YOUR_TAVILY_API_KEY"

# Fallback for demonstration if not set in environment
if "OPENAI_API_KEY" not in os.environ:
    print("WARNING: OPENAI_API_KEY not found in environment variables. Using a placeholder.")
    # In a real scenario, you would raise an error or prompt the user.
    # For this demo, we'll proceed, but the LLM calls might fail without a valid key.
    os.environ["OPENAI_API_KEY"] = "sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"

if "TAVILY_API_KEY" not in os.environ:
    print("WARNING: TAVILY_API_KEY not found in environment variables. Using a placeholder.")
    os.environ["TAVILY_API_KEY"] = "tvly-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"

# Initialize the LLM (using GPT-4o for advanced reasoning capabilities)
llm = ChatOpenAI(model="gpt-4o", temperature=0.7)

# --- 2. Define Tools ---
# Our researcher agent will need a search tool
tavily_tool = TavilySearchResults(max_results=5)
tools = [tavily_tool]

# --- 3. Define Agent States (LangGraph State) ---
# This defines the state that will be passed between agents in our graph.
class AgentState(TypedDict):
    task: str  # The overall task description
    research_notes: str  # Notes gathered by the researcher
    draft: str  # The initial draft by the writer
    final_article: str # The refined article by the editor
    messages: List[BaseMessage] # A history of messages for conversational agents (optional, but good practice)

# --- 4. Define Specialized Agents ---

# Agent 1: Researcher Agent
def create_researcher_agent(llm, tools):
    research_prompt = ChatPromptTemplate.from_messages([
        SystemMessage("You are a diligent research assistant. Your goal is to gather comprehensive information on the given topic using the provided search tools. Focus on factual accuracy and provide concise, relevant notes."),
        HumanMessage("{task}"),
        SystemMessage("Based on the user's request, perform searches and compile key findings. If you have enough information, state 'RESEARCH_COMPLETE'. Otherwise, continue searching.")
    ])
    return create_openai_functions_agent(llm, tools, research_prompt)

researcher_agent_executor = AgentExecutor(agent=create_researcher_agent(llm, tools), tools=tools, verbose=True)

# Agent 2: Writer Agent
def create_writer_agent(llm):
    writer_prompt = ChatPromptTemplate.from_messages([
        SystemMessage("You are a skilled content writer. Your task is to write a well-structured and engaging article based on the provided research notes. Ensure clarity, coherence, and a professional tone. The article should be at least 300 words."),
        HumanMessage("Research Notes: {research_notes}\n\nTask: Write an article about: {task}")
    ])
    return writer_prompt | llm # Using LangChain Expression Language for a simple chain

# Agent 3: Editor Agent
def create_editor_agent(llm):
    editor_prompt = ChatPromptTemplate.from_messages([
        SystemMessage("You are a meticulous editor. Your task is to review and refine the provided article draft for grammar, spelling, clarity, flow, and factual accuracy. Improve the overall quality and ensure it meets professional standards. Provide the final, polished article."),
        HumanMessage("Original Draft: {draft}\n\nTask: Refine the following article based on the original task: {task}")
    ])
    return editor_prompt | llm

# --- 5. Define Graph Nodes (Agent Functions) ---

def research_node(state: AgentState) -> Dict[str, Any]:
    print("---RESEARCHER NODE---")
    result = researcher_agent_executor.invoke({"input": state["task"], "chat_history": state["messages"]})
    # Extract relevant info from the agent's output
    research_notes = result.get("output", "No research notes found.")
    return {"research_notes": research_notes, "messages": state["messages"] + [HumanMessage(content=f"Research complete. Notes: {research_notes}")]}

def write_node(state: AgentState) -> Dict[str, Any]:
    print("---WRITER NODE---")
    draft = create_writer_agent(llm).invoke({"task": state["task"], "research_notes": state["research_notes"]}).content
    return {"draft": draft, "messages": state["messages"] + [HumanMessage(content=f"Draft created. Length: {len(draft.split())} words.")]}

def edit_node(state: AgentState) -> Dict[str, Any]:
    print("---EDITOR NODE---")
    final_article = create_editor_agent(llm).invoke({"task": state["task"], "draft": state["draft"]}).content
    return {"final_article": final_article, "messages": state["messages"] + [HumanMessage(content=f"Article edited. Final length: {len(final_article.split())} words.")]}

# --- 6. Build the LangGraph Workflow ---

workflow = StateGraph(AgentState)

# Add nodes for each agent's function
workflow.add_node("researcher", research_node)
workflow.add_node("writer", write_node)
workflow.add_node("editor", edit_node)

# Define the entry point
workflow.set_entry_point("researcher")

# Define the edges (transitions between nodes)
workflow.add_edge("researcher", "writer")
workflow.add_edge("writer", "editor")
workflow.add_edge("editor", END) # The editor node is the final step

# Compile the graph
app = workflow.compile()

# --- 7. Run the Multi-Agent System ---

# Define the initial task
task_description = "Write a comprehensive article about the latest advancements in quantum computing, focusing on its potential impact on cryptography and AI by 2026."

# Initial state for the graph
initial_state = {
    "task": task_description,
    "research_notes": "",
    "draft": "",
    "final_article": "",
    "messages": [HumanMessage(content=task_description)]
}

print(f"\n--- Starting Multi-Agent Workflow for Task: {task_description} ---\n")

# Run the graph
final_state = app.invoke(initial_state)

print("\n--- Workflow Completed ---\n")
print("\n--- Final Article ---\n")
print(final_state["final_article"])

print("\n--- Research Notes (for context) ---\n")
print(final_state["research_notes"])


### Interpreting the Code Output and Use Cases

When you run the provided code, you'll observe a sequence of operations, each corresponding to a specialized agent performing its designated task:

1.  **Researcher Node**: You'll see output from the `researcher_agent_executor` as it uses the `TavilySearchResults` tool. It will perform web searches based on the `task` and compile `research_notes`. The `verbose=True` setting for the `AgentExecutor` will show the thought process of the agent, including tool calls and observations.
2.  **Writer Node**: Once the researcher is done, the `writer_node` takes the `research_notes` and the original `task` to generate an initial `draft` of the article. There won't be tool calls here, as the writer's job is purely generative based on the input.
3.  **Editor Node**: Finally, the `editor_node` receives the `draft` and the `task`. It then refines, corrects, and polishes the article, producing the `final_article`.

The final output will display the complete, polished article, along with the research notes that informed its creation. This demonstrates a clear, sequential flow where each agent builds upon the work of the previous one.

### Performance Trade-offs and Considerations

While multi-agent orchestration offers significant advantages, it's crucial to understand the trade-offs:

*   **Increased Latency**: Each agent interaction, especially if it involves LLM calls or external tool usage, adds to the overall execution time. A sequential chain like our example will be slower than a single, monolithic agent attempting the same task.
*   **Higher Cost**: More LLM calls generally mean higher API costs. Careful prompt engineering and efficient tool usage are essential to mitigate this.
*   **Complexity**: Designing, debugging, and maintaining a multi-agent system is inherently more complex than a single-agent system. State management, error handling, and ensuring smooth transitions between agents require careful planning.
*   **Orchestration Overhead**: The `LangGraph` framework itself introduces some overhead in managing the state and transitions, though it's generally optimized for performance.

### Typical Use Cases for Multi-Agent Orchestration

Multi-agent systems excel in scenarios requiring diverse expertise, complex decision-making, or iterative refinement:

*   **Automated Content Generation**: Beyond simple articles, this can extend to generating marketing copy, technical documentation, or even creative stories, with agents specializing in research, drafting, SEO optimization, and editing.
*   **Complex Data Analysis**: Agents can specialize in data extraction, cleaning, statistical analysis, visualization, and report generation, collaborating to produce comprehensive insights.
*   **Software Development**: Agents can take on roles like requirements analysis, code generation, testing, and documentation, working together to build software components.
*   **Customer Support & Sales**: Hierarchical agents where a primary agent triages requests and delegates to specialized agents (e.g., technical support, billing, product information) can provide more efficient and accurate responses.
*   **Autonomous Research & Experimentation**: Agents can design experiments, execute simulations, analyze results, and propose next steps, accelerating scientific discovery.
*   **Strategic Planning & Decision Support**: Agents can gather market intelligence, analyze competitor strategies, forecast trends, and propose business strategies to a human decision-maker.

By leveraging multi-agent orchestration, developers can build more capable, robust, and scalable AI systems that can tackle problems far beyond the scope of a single agent.


### Resources

*   **LangChain Documentation - LangGraph**: The official guide to building stateful, multi-actor applications with LangChain. [https://langchain-ai.github.io/langgraph/](https://langchain-ai.github.io/langgraph/)
*   **LangChain Documentation - Agents**: Deep dive into LangChain's agent capabilities and `AgentExecutor`. [https://python.langchain.com/docs/modules/agents/](https://python.langchain.com/docs/modules/agents/)
*   **Tavily Search API**: Learn more about the search tool used in this example. [https://tavily.com/](https://tavily.com/)
*   **OpenAI API Documentation**: Information on `gpt-4o` and other models. [https://platform.openai.com/docs/](https://platform.openai.com/docs/)
*   **Google AI Studio / Gemini API**: Explore Google's LLM offerings for alternative model integrations. [https://ai.google.dev/](https://ai.google.dev/)
*   **Hugging Face Models**: Discover a vast ecosystem of open-source LLMs and models that can be integrated with LangChain. [https://huggingface.co/models](https://huggingface.co/models)
